# 🐛 DebugZero — GRPO Training on Colab

Two modes available:
- **Offline** — builds a static bug bank, trains GRPO on it (fast, simple)
- **Online** *(AZR-inspired)* — model generates its own bugs each iteration (self-play)

**Recommended GPU:** T4 (free tier) or A100 (Colab Pro)

## 1. Setup & Install

In [ ]:
# Clone the repo (skip if already cloned)
import os
if not os.path.exists('DebugZero'):
    !git clone https://github.com/YOUR_USERNAME/DebugZero.git
os.chdir('DebugZero')
print('Working dir:', os.getcwd())

In [ ]:
%%capture
!pip install -q unsloth
!pip install -q trl transformers datasets openenv-core[core] pydantic thefuzz matplotlib bitsandbytes

In [ ]:
# Verify GPU is available
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected! Go to Runtime → Change runtime type → GPU')

## 2. Configuration

In [ ]:
# === Training Mode ===
ONLINE_MODE = False   # False = offline GRPO (recommended to start)
DRY_RUN = False       # True = tiny smoke test (no real model)

# === Online mode settings (ignored if ONLINE_MODE=False) ===
ITERATIONS = 3        # Number of self-play iterations
STEPS_PER_ITER = 30   # GRPO steps per iteration

## 3. Verify Data

In [ ]:
from training.grpo_train import create_dataset

dataset, bug_bank = create_dataset()
print(f'✅ Dataset rows:    {len(dataset)}')
print(f'✅ Training bugs:   {len(bug_bank.train_samples)}')
print(f'✅ Eval bugs:       {len(bug_bank.eval_samples)}')
print(f'\nSample row:')
dataset[0]

## 4. Train

In [ ]:
if ONLINE_MODE:
    from training.online_grpo_train import run_online_workflow
    results = run_online_workflow(
        dry_run=DRY_RUN,
        iterations=ITERATIONS,
        steps_per_iter=STEPS_PER_ITER,
    )
else:
    from training.grpo_train import run_workflow
    results = run_workflow(dry_run=DRY_RUN)

print('\n✅ Training complete!')

## 5. Results

In [ ]:
print('=== Pre-Training ===')
print(f"  Solver:   {results['pre_solver_metrics']}")
print(f"  Proposer: {results['pre_proposer_metrics']}")
print()
print('=== Post-Training ===')
print(f"  Solver:   {results['post_solver_metrics']}")
print(f"  Proposer: {results['post_proposer_metrics']}")

# Online mode: show per-iteration metrics
if ONLINE_MODE and 'iteration_metrics' in results:
    print('\n=== Per-Iteration ===')
    for m in results['iteration_metrics']:
        print(f"  Iter {m['iteration']}: pool={m['pool_size']} ds={m['dataset_size']} "
              f"solver={m['solver']} proposer={m['proposer']}")

In [ ]:
from IPython.display import Image, display

if results.get('plot_path'):
    display(Image(filename=results['plot_path']))
else:
    print('No plot generated.')

## 6. Save Model (Optional)

In [ ]:
# Save to Google Drive (optional)
SAVE_TO_DRIVE = False

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    !cp -r debugzero_model /content/drive/MyDrive/debugzero_model
    print('✅ Model saved to Google Drive')
else:
    print('Model saved locally at: debugzero_model/')
    print('Set SAVE_TO_DRIVE = True above to persist to Google Drive.')